In [ ]:
import numpy as np 
import pandas as pd 

df = pd.read_csv("/home/rasool-estiri/artificial-intelligence/project/real-estate/data/Divar.csv")
city_classification = pd.read_csv("/home/rasool-estiri/artificial-intelligence/project/real-estate/data/iran_city_classification.csv")


In [ ]:
df.columns.tolist()
df['cat3_slug'].value_counts()

In [ ]:
num_cols = [
    'building_size', 'floor', 
    'rooms_count', 'total_floors_count', 
    'unit_per_floor', 'construction_year'
]

In [ ]:
df[num_cols].head()
#unit per floor : تعداد واحدهای هر طبقه 

In [ ]:
df[num_cols].dtypes

In [ ]:
trans_table = str.maketrans('۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩', '01234567890123456789')

df['construction_year'] = df['construction_year'].astype(str).str.translate(trans_table)

# با این کار "قبل از 1370" تبدیل به "1370" می‌شود
df['construction_year'] = df['construction_year'].str.extract(r'(\d+)')[0]

# ۴. تبدیل نهایی به نوع عددی (با پوشش دادن مقادیر مفقود احتمالی)
df['construction_year'] = pd.to_numeric(df['construction_year'], errors='coerce')

# ۵. در صورت نیاز به نوع Int64 (پشتیبانی از Int همراه با NaN)
df['construction_year'] = df['construction_year'].astype('Int64')

df["construction_year"].value_counts()

# نمایش تمام مقادیر یکتا همراه با تعداد تکرار
print(df['construction_year'].value_counts(dropna=False))

In [ ]:
print(df["building_size"].isnull().sum() / len(df["building_size"]) * 100 )
df["building_size"].value_counts()


In [ ]:
# rooms count 
rooms_map = {
    'بدون اتاق': 0,
    'یک': 1,
    'دو': 2,
    'سه': 3,
    'چهار': 4,
    'پنج یا بیشتر': 5
}

df['rooms_count'] = df['rooms_count'].map(rooms_map)

# در صورت وجود مقدار مفقود (NaN)، تبدیل به نوع عددی Int64 پشتیبانی‌کننده از NaN
df['rooms_count'] = df['rooms_count'].astype('Int64')

In [ ]:
#floor 
df['floor'] = (
    df['floor']
    .astype(str)
    .str.strip()
    .str.replace('30+', '30', regex=False)  # تبدیل +30 به 30
    .str.replace(r'\.0$', '', regex=True)  # تبدیل 2.0 به 2
)

df['floor'] = pd.to_numeric(df['floor'], errors='coerce').astype('Int64')

In [ ]:
#totoal_floor-count 

df['total_floors_count'] = (
    df['total_floors_count']
    .astype(str)
    .str.strip()
    .str.replace('unselect', '', regex=False)
    .str.replace('30+', '30', regex=False)     
    .str.replace(r'\.0$', '', regex=True)   
)

df['total_floors_count'] = pd.to_numeric(df['total_floors_count'], errors='coerce').astype('Int64')

In [ ]:
#unit_per_floor

df['unit_per_floor'] = (
    df['unit_per_floor']
    .astype(str)
    .str.strip()
    .str.replace('more_than_8', '9', regex=False)
    .str.replace('unselect', '', regex=False)
)

df['unit_per_floor'] = pd.to_numeric(df['unit_per_floor'], errors='coerce').astype('Int64')

In [ ]:
#land_size
# بیشتر از 81 درصد از داده های این ستون نامشخص هستند
# تصمیم شد که یک به وسیله این یک ویژگی جدید بسازیم 
#میتونیم اینجوری برداشت کنیم در خانه هایی که این ویژگی خالی است یک خانه آپارتمانی  هست و چون زمین متعلق به آپارتمان هست٬ متراژ اون رو وارد نمی کنیم
# ساخت ویژگی باینری و حذف ستون اصلی

df['has_land'] = df['land_size'].notnull().astype('Int64')
df = df.drop(columns=['land_size'])

In [ ]:
# ۱. برای املاک مسکونی (residential): متراژ بالای ۱,۰۰۰ متر معمولاً غیرمنطقی است
residential_mask = df['cat2_slug'].str.contains('residential', na=False)
df.loc[residential_mask & ((df['building_size'] < 10) | (df['building_size'] > 1000)), 'building_size'] = np.nan

# ۲. برای املاک تجاری/اداری/خدماتی (commercial): سقف متراژ را روی ۲۰,۰۰۰ متر می‌گذاریم
commercial_mask = df['cat2_slug'].str.contains('commercial', na=False)
df.loc[commercial_mask & ((df['building_size'] < 10) | (df['building_size'] > 20000)), 'building_size'] = np.nan

# ۳. برای سایر دسته‌ها (زمین/کلنگی، اجاره کوتاه مدت و ...): متراژهای پرت کلی (زیر ۱۰ یا بالای ۱۰,۰۰۰ متر) -> NaN
other_mask = ~residential_mask & ~commercial_mask
df.loc[other_mask & ((df['building_size'] < 10) | (df['building_size'] > 10000)), 'building_size'] = np.nan

In [ ]:
print("حداقل متراژ:", df['building_size'].min())
print("حداکثر متراژ:", df['building_size'].max())

In [ ]:
df["building_size"].shape
df["building_size"].isnull().sum() / len(df["building_size"]) * 100

In [ ]:
res_outliers = df[(df['cat2_slug'].str.contains('residential', na=False)) & (df['building_size'] > 1000)]
print("تعداد پرت‌های مسکونی باقی‌مانده:", len(res_outliers))